# 05 — PyCaret AutoML + Seasonal Naive baseline

Produces the forecasts of the two non-foundation models required by the
multi-model comparison (notebook 06):

- **PyCaret AutoML** — trains ~30 classical models per series and keeps the best one.
- **Seasonal Naive** — the `s=12` baseline that defines the MASE denominator.

**Environment:** main `.venv` (see `requirements.txt`).

**No executed outputs** — pedagogical template. PyCaret is the slowest model
(orders of magnitude more than any foundation model).

## 1. Configuration

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import load_parquet, filter_period, build_series
from src.metrics import seasonal_naive

PARQUET_PATH = REPO_ROOT / 'data' / 'anonymized_series.parquet'
OUTPUT_PYCARET = REPO_ROOT / 'outputs' / 'pycaret'
OUTPUT_BASELINE = REPO_ROOT / 'outputs' / 'baseline'
OUTPUT_PYCARET.mkdir(parents=True, exist_ok=True)
OUTPUT_BASELINE.mkdir(parents=True, exist_ok=True)

HORIZON = 12

## 2. Loading the series

In [ ]:
df = load_parquet(PARQUET_PATH)
df = filter_period(df, 2020, 2024)
series = build_series(df, min_months=24)
print(f'Series: {len(series)}')

## 3. Seasonal Naive baseline

The forecast for month `t + h` is the value observed at `t + h - 12`. It runs
in well under a millisecond per series.

In [ ]:
pred_rows = []
for sid, s in series.items():
    train, test = s.split_train_test(horizon=HORIZON)
    y_pred = seasonal_naive(train, HORIZON)
    for h in range(HORIZON):
        pred_rows.append({
            'series_id': sid,
            'horizon_month': h + 1,
            'y_true': float(test[h]),
            'pred_seasonal_naive': float(y_pred[h]),
        })
pd.DataFrame(pred_rows).to_csv(OUTPUT_BASELINE / 'predictions_seasonal_naive.csv', index=False)
print('Seasonal Naive done.')

## 4. PyCaret AutoML (best model per series)

`compare_models()` cross-validates ~30 classical models on temporal blocks and
keeps the best one. It requires at least 36 months of history for the
annual-seasonality models; shorter series are skipped and reported.

In [ ]:
from pycaret.time_series import TSForecastingExperiment

pred_rows = []
skipped = []

for sid, s in series.items():
    train, test = s.split_train_test(horizon=HORIZON)
    if len(train) < 36:
        skipped.append(sid)
        continue

    train_series = pd.Series(
        train,
        index=pd.PeriodIndex(s.dates[:-HORIZON], freq='M'),
        name='value',
    )

    exp = TSForecastingExperiment()
    exp.setup(data=train_series, fh=HORIZON, seasonal_period=12,
              session_id=42, verbose=False)
    best = exp.compare_models()
    final = exp.finalize_model(best)
    forecast = exp.predict_model(final, fh=HORIZON)
    y_pred = np.clip(np.asarray(forecast).ravel()[:HORIZON], 0, None)

    for h in range(HORIZON):
        pred_rows.append({
            'series_id': sid,
            'horizon_month': h + 1,
            'y_true': float(test[h]),
            'pred_pycaret': float(y_pred[h]),
        })

pd.DataFrame(pred_rows).to_csv(OUTPUT_PYCARET / 'predictions_pycaret.csv', index=False)
print(f'PyCaret done. Skipped (history < 36 months): {len(skipped)}')